# Credit Agreement Analyzer - Phase 3 Validation
Testing retrieval layer (embedder, vector store, BM25, hybrid) against a real credit agreement.

In [ ]:
import contextlib
import time
from collections import Counter
from pathlib import Path

from credit_analyzer.processing.chunker import Chunker
from credit_analyzer.processing.definitions import DefinitionsParser
from credit_analyzer.processing.pdf_extractor import PDFExtractor
from credit_analyzer.processing.section_detector import SectionDetector
from credit_analyzer.retrieval.bm25_store import BM25Store
from credit_analyzer.retrieval.embedder import Embedder
from credit_analyzer.retrieval.hybrid_retriever import HybridRetriever
from credit_analyzer.retrieval.vector_store import VectorStore

PDF_PATH = Path(r"../credit_agreement.pdf")
CHROMA_DIR = str(Path(r"../chroma_validation"))

## Step 0: Rebuild Phase 1 Outputs
Re-run extraction, section detection, definitions, and chunking.

In [ ]:
extractor = PDFExtractor()
doc = extractor.extract(PDF_PATH)
print(f"Pages: {doc.total_pages}")

detector = SectionDetector()
sections = detector.detect_sections(doc)
print(f"Sections: {len(sections)}")

defn_sections = [s for s in sections if s.section_type == "definitions"]
if not defn_sections:
    msg = "No definitions section found"
    raise RuntimeError(msg)
parser = DefinitionsParser()
defn_index = parser.parse(defn_sections[0])
print(f"Defined terms: {len(defn_index.definitions)}")

chunker = Chunker()
chunks = chunker.chunk_document(sections, defn_index)
print(f"Chunks: {len(chunks)}")

## Step 1: Embedder
Embed all chunks and verify dimensions, timing, and basic sanity.

In [ ]:
embedder = Embedder()
print(f"Model dimension: {embedder.dimension}")

start = time.time()
embeddings = embedder.embed([c.text for c in chunks])
elapsed = time.time() - start

print(f"Embedded {len(embeddings)} chunks in {elapsed:.1f}s")
print(f"Embedding dim: {len(embeddings[0])}")
print(f"Rate: {len(embeddings) / elapsed:.0f} chunks/sec")

In [ ]:
# Sanity: query embedding should have same dimension
q_emb = embedder.embed_query("What is the restricted payments basket?")
print(f"Query embedding dim: {len(q_emb)}")
assert len(q_emb) == len(embeddings[0]), "Dimension mismatch!"

## Step 2: Vector Store
Store chunks + embeddings in ChromaDB and test retrieval.

In [ ]:
store = VectorStore(persist_directory=CHROMA_DIR)
# Delete any stale collection from previous runs
with contextlib.suppress(Exception):
    store.delete_collection("ribbon_test")
store.create_collection("ribbon_test")
store.add_chunks("ribbon_test", chunks, embeddings)
print(f"Stored {len(chunks)} chunks in ChromaDB")
print(f"Documents in store: {store.list_documents()}")

In [ ]:
# Test: semantic search for restricted payments
q = "What is the restricted payments basket?"
q_emb = embedder.embed_query(q)
results = store.search("ribbon_test", q_emb, top_k=5)

print(f"Query: {q}")
print(f"Results: {len(results)}\n")
for r in results:
    print(f"  Score: {r.score:.3f} | {r.chunk.section_id} - {r.chunk.section_title} ({r.chunk.section_type})")
    print(f"  Text: {r.chunk.text[:150]}...\n")

In [ ]:
# Test: filtered search within negative covenants only
q = "How much incremental debt can the borrower incur?"
q_emb = embedder.embed_query(q)
results = store.search("ribbon_test", q_emb, top_k=5, section_filter="negative_covenants")

print(f"Query: {q}")
print("Filter: negative_covenants")
print(f"Results: {len(results)}\n")
for r in results:
    print(f"  Score: {r.score:.3f} | {r.chunk.section_id} - {r.chunk.section_title}")
    print(f"  Text: {r.chunk.text[:150]}...\n")

## Step 3: BM25 Store
Test keyword search for exact terms, dollar amounts, and ratios.

In [ ]:
bm25 = BM25Store()
bm25.build_index(chunks)
print(f"BM25 index built over {len(chunks)} chunks")

In [ ]:
# Test: exact dollar amount search
for q in ["$50,000,000", "4.75:1.00", "Incremental Facility", "Restricted Payments"]:
    results = bm25.search(q, top_k=3)
    print(f"Query: {q}")
    if results:
        for r in results:
            print(f"  Score: {r.score:.3f} | {r.chunk.section_id} - {r.chunk.section_title} ({r.chunk.chunk_type})")
    else:
        print("  No results")
    print()

In [ ]:
# Test: BM25 with section filter
results = bm25.search("Restricted Payments", top_k=5, section_filter="negative_covenants")
print(f"BM25 'Restricted Payments' filtered to negative_covenants: {len(results)} results")
for r in results:
    print(f"  Score: {r.score:.3f} | {r.chunk.section_id} - {r.chunk.section_title}")

## Step 4: Hybrid Retriever
Test combined vector + BM25 retrieval with definition injection.

In [ ]:
retriever = HybridRetriever(
    vector_store=store,
    bm25_store=bm25,
    embedder=embedder,
    definitions_index=defn_index,
)

In [ ]:
# Test query 1: Incremental debt capacity (your core use case)
result = retriever.retrieve(
    query="How much incremental debt can the borrower incur?",
    document_id="ribbon_test",
    top_k=5,
)

print("Query: How much incremental debt can the borrower incur?")
print(f"Chunks returned: {len(result.chunks)}\n")
for hc in result.chunks:
    print(f"  Score: {hc.score:.3f} | Source: {hc.source:6s} | {hc.chunk.section_id} - {hc.chunk.section_title}")
    print(f"  Text: {hc.chunk.text[:200]}...\n")

print(f"Injected definitions ({len(result.injected_definitions)}):")
for term, defn in result.injected_definitions.items():
    print(f"  {term}: {defn[:100]}...")

In [ ]:
# Test query 2: Financial covenants
result = retriever.retrieve(
    query="What are the financial covenant test levels?",
    document_id="ribbon_test",
    top_k=5,
)

print("Query: What are the financial covenant test levels?")
print(f"Chunks returned: {len(result.chunks)}\n")
for hc in result.chunks:
    print(f"  Score: {hc.score:.3f} | Source: {hc.source:6s} | {hc.chunk.section_id} - {hc.chunk.section_title}")
    print(f"  Type: {hc.chunk.chunk_type}")
    print(f"  Text: {hc.chunk.text[:200]}...\n")

print(f"Injected definitions ({len(result.injected_definitions)}):")
for term in result.injected_definitions:
    print(f"  {term}")

In [ ]:
# Test query 3: Specific dollar amount (BM25 should shine here)
result = retriever.retrieve(
    query="What is the revolving commitment amount?",
    document_id="ribbon_test",
    top_k=5,
)

print("Query: What is the revolving commitment amount?")
print(f"Chunks returned: {len(result.chunks)}\n")
for hc in result.chunks:
    print(f"  Score: {hc.score:.3f} | Source: {hc.source:6s} | {hc.chunk.section_id} - {hc.chunk.section_title}")
    print(f"  Text: {hc.chunk.text[:200]}...\n")

In [ ]:
# Test query 4: With section filter (report generation pattern)
result = retriever.retrieve(
    query="What restricted payments are permitted?",
    document_id="ribbon_test",
    top_k=5,
    section_filter="negative_covenants",
)

print("Query: What restricted payments are permitted? (filtered: negative_covenants)")
print(f"Chunks returned: {len(result.chunks)}\n")
for hc in result.chunks:
    print(f"  Score: {hc.score:.3f} | Source: {hc.source:6s} | {hc.chunk.section_id} - {hc.chunk.section_title}")
    print(f"  Text: {hc.chunk.text[:200]}...\n")

print(f"Injected definitions ({len(result.injected_definitions)}):")
for term in result.injected_definitions:
    print(f"  {term}")

In [ ]:
# Check source distribution across all test queries
test_queries = [
    "How much incremental debt can the borrower incur?",
    "What are the financial covenant test levels?",
    "What is the revolving commitment amount?",
    "What restricted payments are permitted?",
    "Who are the administrative agents?",
    "What are the mandatory prepayment triggers?",
]

source_counts: Counter[str] = Counter()
for q in test_queries:
    r = retriever.retrieve(q, "ribbon_test", top_k=5)
    for hc in r.chunks:
        source_counts[hc.source] += 1

print("Source distribution across test queries:")
total = sum(source_counts.values())
for source, count in source_counts.most_common():
    print(f"  {source}: {count} ({count/total*100:.0f}%)")

## Cleanup

In [ ]:
store.delete_collection("ribbon_test")
print("Cleaned up test collection")